In [12]:
import csv
import numpy as np

from helpers import create_csv_submission, load_csv_data
from modeling import fit_final_ridge, select_ridge_by_oof_f1


In [13]:
data_path = 'dataset'
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(data_path)

with open(f'{data_path}/x_train.csv', newline='') as csv_file:
    feature_names = next(csv.reader(csv_file))[1:]

print(f'Train: {x_train.shape}; test: {x_test.shape}')
print(f'Positive training-label rate: {np.mean(y_train == 1):.2%}')


Train: (328135, 321); test: (109379, 321)
Positive training-label rate: 8.83%


In [14]:
missing_threshold = 0.05
cv_lambdas = (1e-6, 1e-5, 1e-4, 1e-3, 1e-2)

model_selection = select_ridge_by_oof_f1(
    x_train,
    y_train,
    train_ids,
    feature_names,
    missing_threshold=missing_threshold,
    lambdas=cv_lambdas,
    fold_count=3,
    random_state=123,
)

print('Fold results:')
for fold_result in model_selection['fold_reports']:
    print(fold_result)
print('Lambda results (OOF AUC and best F1):')
for result in model_selection['lambda_results']:
    print(
        f"lambda={result['lambda']:.0e}, AUC={result['auc']:.4f}, "
        f"best F1={result['best_f1']['f1']:.4f} "
        f"at threshold={result['best_f1']['threshold']:.6f}"
    )

best_lambda = model_selection['best_lambda']
best_threshold = model_selection['best_threshold']
print(f'F1-selected lambda: {best_lambda:.0e}')
print(f'F1-selected threshold: {best_threshold:.6f}')
print('OOF F1 is used for selection and is therefore an optimistic estimate, not a final generalization score.')


Fold results:
{'fold': 1, 'fit_rows': 218756, 'validation_rows': 109379, 'features': 175}
{'fold': 2, 'fit_rows': 218757, 'validation_rows': 109378, 'features': 175}
{'fold': 3, 'fit_rows': 218757, 'validation_rows': 109378, 'features': 175}
Lambda results (OOF AUC and best F1):
lambda=1e-06, AUC=0.8473, best F1=0.4108 at threshold=-0.561574
lambda=1e-05, AUC=0.8474, best F1=0.4106 at threshold=-0.559279
lambda=1e-04, AUC=0.8469, best F1=0.4088 at threshold=-0.561372
lambda=1e-03, AUC=0.8465, best F1=0.4078 at threshold=-0.565800
lambda=1e-02, AUC=0.8473, best F1=0.4078 at threshold=-0.595438
F1-selected lambda: 1e-06
F1-selected threshold: -0.561574
OOF F1 is used for selection and is therefore an optimistic estimate, not a final generalization score.


In [15]:
final_model = fit_final_ridge(
    x_train,
    x_test,
    y_train,
    train_ids,
    test_ids,
    feature_names,
    lambda_value=best_lambda,
    threshold=best_threshold,
    missing_threshold=missing_threshold,
)

submission_path = 'submission_f1_ridge.csv'
create_csv_submission(test_ids, final_model['predictions'], submission_path)

assert len(final_model['predictions']) == len(test_ids)
assert set(np.unique(final_model['predictions'])).issubset({-1, 1})
assert np.isfinite(final_model['scores']).all()

print(f"Refit on all {len(y_train)} labeled rows.")
print(f"Test predictions: {len(test_ids)}; positive predictions: {final_model['positive_predictions']}.")
print(f"Submission created: {submission_path}")


Refit on all 328135 labeled rows.
Test predictions: 109379; positive predictions: 14791.
Submission created: submission_f1_ridge.csv
